# 04 — Model Training
**E-Waste Toxic Gas Detection System — ML Pipeline**

**Purpose:** Train all 4 ML models (Random Forest, SVM, Decision Tree, Naive Bayes) on the preprocessed dataset.

**Author:** Sanjula Madushanka | Final Year Research Y4S2

In [ ]:
import pandas as pd
import numpy as np
import time
import joblib
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

DATA_DIR  = Path('../datasets/processed/train_test_split')
MODEL_DIR = Path('../models')
MODEL_DIR.mkdir(exist_ok=True)

# Load data
X_train = pd.read_csv(DATA_DIR / 'X_train.csv').values
X_test  = pd.read_csv(DATA_DIR / 'X_test.csv').values
y_train = pd.read_csv(DATA_DIR / 'y_train.csv').values.ravel()
y_test  = pd.read_csv(DATA_DIR / 'y_test.csv').values.ravel()
le      = joblib.load(MODEL_DIR / 'label_encoder.pkl')

print(f'Training set:  {X_train.shape}')
print(f'Test set:      {X_test.shape}')
print(f'Classes:       {list(le.classes_)}')
print(f'n_classes:     {len(le.classes_)}')

## 4.1 Define All Models

In [ ]:
# ============================================================
# MODEL DEFINITIONS — With academic justification comments
# ============================================================

MODELS = {
    'Random Forest': RandomForestClassifier(
        n_estimators=200,        # 200 trees — balanced accuracy vs speed
        max_depth=None,          # Allow full depth (will tune in notebook 06)
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',     # sqrt(7) ≈ 2.6 → 3 features per split
        class_weight='balanced', # Handle class imbalance
        random_state=42,
        n_jobs=-1                # Use all CPU cores
    ),
    'SVM': SVC(
        kernel='rbf',            # Radial basis function — best for non-linear sensor data
        C=10,                    # Regularisation parameter
        gamma='scale',           # 1/(n_features × X.var())
        class_weight='balanced',
        probability=True,        # Enable predict_proba (for confidence %)
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,            # Limit depth to prevent overfitting
        criterion='gini',        # Gini impurity
        min_samples_split=5,
        class_weight='balanced',
        random_state=42
    ),
    'Naive Bayes': GaussianNB(
        var_smoothing=1e-9       # Gaussian smoothing parameter
    )
}

print('=== MODELS TO TRAIN ===')
for name, model in MODELS.items():
    print(f'  {name}')
print()
print('Justification:')
print('  Random Forest: Ensemble method — robust to noise, handles high-dimensional data')
print('  SVM:           Strong for small-to-medium datasets with clear margins')
print('  Decision Tree: Interpretable — can visualise decision rules')
print('  Naive Bayes:   Probabilistic baseline — fast, works well with independent features')

## 4.2 Train All Models with Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('Training models with 5-Fold Stratified Cross-Validation...\n')
print(f'{"Model":<20} {"Train Acc":>10} {"CV Mean":>10} {"CV Std":>10} {"Time":>10}')
print('-' * 65)

for name, model in MODELS.items():
    start = time.perf_counter()
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv,
                                scoring='accuracy', n_jobs=-1)
    
    # Final fit on full training set
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    
    elapsed = time.perf_counter() - start
    
    results[name] = {
        'model':       model,
        'cv_scores':   cv_scores,
        'cv_mean':     cv_scores.mean(),
        'cv_std':      cv_scores.std(),
        'train_acc':   train_acc,
        'train_time':  elapsed
    }
    
    print(f'{name:<20} {train_acc*100:>9.2f}% {cv_scores.mean()*100:>9.2f}% '
          f'{cv_scores.std()*100:>9.2f}% {elapsed:>9.2f}s')

print('\n✅ All models trained successfully!')

## 4.3 Save All Trained Models

In [ ]:
MODEL_FILES = {
    'Random Forest': 'random_forest_v1.pkl',
    'SVM':           'svm_v1.pkl',
    'Decision Tree': 'decision_tree_v1.pkl',
    'Naive Bayes':   'naive_bayes_v1.pkl',
}

print('=== SAVING MODELS ===')
for name, filename in MODEL_FILES.items():
    path = MODEL_DIR / filename
    joblib.dump(results[name]['model'], path)
    size_kb = path.stat().st_size / 1024
    print(f'  ✅ {filename} ({size_kb:.1f} KB)')

print(f'\nAll models saved to: {MODEL_DIR}')

## 4.4 Cross-Validation Score Comparison Chart

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
SAVE_DIR = Path('../results')

model_names = list(results.keys())
cv_means    = [results[m]['cv_mean'] * 100 for m in model_names]
cv_stds     = [results[m]['cv_std']  * 100 for m in model_names]

colors = ['#ef4444', '#3b82f6', '#f59e0b', '#22c55e']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(model_names, cv_means,
              yerr=cv_stds, capsize=5,
              color=colors, edgecolor='white', linewidth=0.8,
              error_kw={'linewidth': 2, 'color': '#1e293b'})

for bar, mean, std in zip(bars, cv_means, cv_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.5,
            f'{mean:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 110)
ax.set_ylabel('5-Fold CV Accuracy (%)', fontsize=11)
ax.set_title('Figure 9: 5-Fold Cross-Validation Accuracy Comparison\n(Error bars = ±1 std)',
             fontsize=13, fontweight='bold')
ax.axhline(90, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='90% baseline')
ax.legend()
plt.tight_layout()
plt.savefig(SAVE_DIR / 'fig9_cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

In [ ]:
best_model_name = max(results, key=lambda m: results[m]['cv_mean'])
best_cv_acc     = results[best_model_name]['cv_mean']

print('=' * 60)
print('TRAINING SUMMARY')
print('=' * 60)
for name in results:
    r = results[name]
    marker = '★ BEST' if name == best_model_name else ''
    print(f'{name:<20}: CV={r["cv_mean"]*100:.2f}% (±{r["cv_std"]*100:.2f}%)  {marker}')

print()
print(f'Best model: {best_model_name} (CV Accuracy = {best_cv_acc*100:.2f}%)')
print()
print('✅ Notebook 04 complete — proceed to 05_model_evaluation.ipynb')